In [10]:
import pandas as pd
import requests
import csv
from tqdm import tqdm
import spacy
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
from skrub import deduplicate
import openai
from transformers import pipeline
import asyncio
import json
import os
from tqdm.auto import tqdm
from openai import AsyncOpenAI
import re
import unicodedata

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/carolyncullen/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
openai.api_key = ''
aclient = AsyncOpenAI(api_key=openai.api_key)

In [12]:
nlp = spacy.load("en_core_web_sm")

In [13]:
pipe = pipeline("text-classification", model="papluca/xlm-roberta-base-language-detection")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1966.98it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: papluca/xlm-roberta-base-language-detection
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
immigration_keywords = {'exact_unigrams': {'ins'},
          'exact_bigrams': {'foreign born',
                             'native born',
                             'ellis island',
                             'foreign worker', 'foreign workers',
                             'foreign labor', 'foreign laborers',
                             'guest worker', 'guest workers',
                             'permanent resident', 'permanent residents', 'permanent residency',
                             'work permit', 'work permits',
                             'day labor', 'day laborer', 'day laborers',
                             'seasonal worker', 'seasonal workers',
                             'low skilled', 'high skilled',
                             'dream act'
                             },
          'p2': {'j1', 'jl'},
          'p3': {'h1b', 'hlb', 'i94', 'h2b', 'dhs', 'ice'},
          'p4': {'visa', 'asyl', 'daca'},
          'p5': {'undoc', 'migra', 'alien', 'quota', 'cuban', 'irish', 'emigr', 'i.n.s', 'd.h.s',
                  'imnig', 'inmig', 'immlg', 'carib', 'uscis', 'asian', 'i.c.e'},
          'p6': {'border', 'deport', 'hispan', 'amnest', 'coolie', 'mongol', 'i.c.e.'},
          'p7': {'customs', 'everify', 'haitian', 'refugee', 'refuges', 'italian', 'mexican', 'bracero'},
          'p8': {'nonrefug', 'foreigne', 'newcomer', 'arrivals', 'dreamers', 'jamaican', 'multicul',
                  'detainee', 'nonquota', 'subquota', 'resettle', 'honduran', 'nonimmig', 'illegals', 'filipino'},
          'p9': {'naturaliz', 'citizensh', 'citizensb', 'nonreside',
                 'detention', 'illegally', 'colombian', 'nonhispan'},
          'p10': {'nativeborn', 'americaniz', 'farmworker', 'lowskilled',
                 'salvadoran', 'guatemalan', 'nicaraguan', 'dominicans'},
          'p11': {'unnaturaliz', 'foreignborn', 'highskilled', 'salvadorian'},
          'seven_letter_exclude': {'alienat', 'quotati', 'borderl', 'caribou'}
}

In [15]:
def match_tokens(tokens, query_terms):
    """
    Determine if a set of tokens matches a set of query terms
    :param tokens: a list of tokens
    :param query_terms: a set of query terms from query_terms.py
    :return: the set of matches
    """

    tokens = [t.lower() for t in tokens]

    # look for an exact unigram
    overlap = list(set(tokens).intersection(query_terms['exact_unigrams']))
    if len(overlap) > 0:
        return True

    # look for exact bigram matches
    bigrams = [tokens[i-1] + ' ' + tokens[i] for i in range(1, len(tokens))]
    overlap = list(set(bigrams).intersection(query_terms['exact_bigrams']))
    if len(overlap) > 0:
        return True

    # look for a particular 5-gram anywhere
    text = ' '.join(tokens)
    if 'immig' in text.lower():
        return True

    # progressively compare tokens to prefixes
    prefixes = set([t[:11] for t in tokens])
    if len(prefixes.intersection(query_terms['p11'])) > 0:
        return True
    prefixes = set([t[:10] for t in prefixes])
    if len(prefixes.intersection(query_terms['p10'])) > 0:
        return True
    prefixes = set([t[:9] for t in prefixes])
    if len(prefixes.intersection(query_terms['p9'])) > 0:
        return True
    prefixes = set([t[:8] for t in prefixes])
    if len(prefixes.intersection(query_terms['p8'])) > 0:
        return True
    prefixes = set([t[:7] for t in prefixes]) - query_terms['seven_letter_exclude']
    if len(prefixes.intersection(query_terms['p7'])) > 0:
        return True
    prefixes = set([t[:6] for t in prefixes])
    if len(prefixes.intersection(query_terms['p6'])) > 0:
        return True
    prefixes = set([t[:5] for t in prefixes])
    if len(prefixes.intersection(query_terms['p5'])) > 0:
        return True
    prefixes = set([t[:4] for t in prefixes])
    if len(prefixes.intersection(query_terms['p4'])) > 0:
        return True
    prefixes = set([t[:3] for t in prefixes])
    if len(prefixes.intersection(query_terms['p3'])) > 0:
        return True
    prefixes = set([t[:2] for t in prefixes])
    if len(prefixes.intersection(query_terms['p2'])) > 0:
        return True

    return False

In [16]:
def skrub_network(network_df, after_skrub):
    sample_length = len(network_df)
    chunk_size = 500
    current_index = 0

    while current_index <= sample_length - 1:
        subset = network_df[current_index: current_index + chunk_size].reset_index()
        
        deduplicated_series = deduplicate(subset['text'])
        deduplicated_series.index = subset.index 
        subset['text_cleaned'] = deduplicated_series

        for i in range(len(subset)):
            after_skrub["date"].append(subset.loc[i]["date"])
            after_skrub["network"].append(subset.loc[i]["network"])
            after_skrub["duration(s)"].append(subset.loc[i]["duration(s)"])
            after_skrub["tag"].append(subset.loc[i]["tag"])
            after_skrub["text"].append(subset.loc[i]["text"])
            after_skrub["text_cleaned"].append(subset.loc[i]["text_cleaned"])

        current_index += chunk_size

    return after_skrub

In [121]:
import os
import re
import json
import asyncio
import pandas as pd
from tqdm.auto import tqdm
from openai import AsyncOpenAI

BAD_CHAR_PATTERN = r"[ÂÃ�_|~*^`]"
MULTISPACE_PATTERN = r"\s+"
WEIRD_PUNCT_PATTERN = r"[^\w\s:/&'.,!?%\-]"


def cheap_clean_text(s):
    if pd.isna(s):
        return s
    s = str(s)

    s = re.sub(BAD_CHAR_PATTERN, " ", s)
    s = re.sub(WEIRD_PUNCT_PATTERN, " ", s)
    s = re.sub(r"\s+([.,!?])", r"\1", s)
    s = re.sub(MULTISPACE_PATTERN, " ", s).strip()

    return s


def _strip_json_fences(s: str) -> str:
    s = (s or "").strip()

    if s.startswith("```"):
        parts = s.split("```")
        if len(parts) >= 2:
            s = parts[1].strip()

        if s.lower().startswith("json"):
            s = s[4:].strip()

    return s


def _parse_llm_list(raw: str, expected_len: int):
    if raw is None:
        return None

    try:
        cleaned_raw = _strip_json_fences(raw)
        data = json.loads(cleaned_raw)
    except Exception:
        return None

    if not isinstance(data, list):
        return None

    parsed = []
    for item in data:
        if item is None:
            parsed.append("")
        elif isinstance(item, str):
            parsed.append(item.strip())
        else:
            return None

    if len(parsed) < expected_len:
        parsed = parsed + [""] * (expected_len - len(parsed))
    elif len(parsed) > expected_len:
        parsed = parsed[:expected_len]

    return parsed


def map_cleaned_list_to_texts(batch_texts, cleaned_list):
    if not isinstance(cleaned_list, list):
        return None
    if len(cleaned_list) != len(batch_texts):
        return None

    mapping = {}
    for original, cleaned in zip(batch_texts, cleaned_list):
        if not isinstance(cleaned, str):
            return None
        mapping[original] = cleaned.strip()

    return mapping


async def _clean_spelling_batch_async(
    texts,
    model="gpt-4o-mini",
    max_retries=5,
):
    """
    Send one batch to the model and return a parsed list[str] or None.
    """
    text_list = [str(t) for t in texts]

    prompt = (
        "You are cleaning OCR-style TV news chyron text.\n"
        "Return ONLY a valid JSON array of strings.\n"
        "Return exactly one output string for each input string.\n"
        "The output array must have the same number of items and same order as the input.\n"
        "Never omit an item. Never add an item. Never merge items.\n\n"
        "For each input:\n"
        "- Clean OCR junk and obvious misspellings.\n"
        "- If the intended chyron is not plausibly about immigration, migrants, immigrants, asylum, deportation, refugee issues, the border, ICE, CBP, visas, or immigration policy, return \"\" for that item.\n"
        "- If uncertain, return \"\".\n\n"
        'Example output: ["text1", "", "text3"]\n\n'
        f"Input texts:\n{json.dumps(text_list, ensure_ascii=False)}"
    )

    last_raw = None

    for attempt in range(max_retries):
        try:
            resp = await aclient.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Return only valid JSON. "
                            "Output must be a JSON array of strings."
                        ),
                    },
                    {"role": "user", "content": prompt},
                ],
                response_format={"type": "text"},
            )

            raw = resp.choices[0].message.content
            last_raw = raw

            parsed = _parse_llm_list(raw, expected_len=len(text_list))
            if parsed is not None:
                return parsed

        except Exception:
            pass

        # small retry pause
        await asyncio.sleep(0.75 * (attempt + 1))

    # optional debug
    print("\nFAILED RAW OUTPUT:\n", last_raw)
    return None


async def _run_batches_concurrently(batches, model="gpt-4o-mini", concurrency=8):
    sem = asyncio.Semaphore(concurrency)

    async def worker(batch_texts):
        async with sem:
            cleaned_list = await _clean_spelling_batch_async(
                batch_texts,
                model=model
            )
            local_map = map_cleaned_list_to_texts(batch_texts, cleaned_list)

            if local_map is None:
                return {t: t for t in batch_texts}, "fallback"

            return local_map, None

    tasks = [asyncio.create_task(worker(b)) for b in batches]

    mapping = {}
    errors = 0

    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        local_map, err = await coro
        mapping.update(local_map)
        if err is not None:
            errors += 1

    return mapping, errors


def clean_dataframe_with_llm_concurrent(
    df,
    text_col="text_cleaned",
    out_col="llm_text",
    batch_size=15,
    concurrency=8,
    model="gpt-4o-mini",
):
    df = df.copy()
    df[text_col] = df[text_col].astype(str).str.strip()
    df[text_col] = df[text_col].apply(cheap_clean_text)

    uniques = df[text_col].dropna().unique().tolist()
    batches = [uniques[i:i + batch_size] for i in range(0, len(uniques), batch_size)]

    if not batches:
        df[out_col] = df[text_col]
        return df

    async def runner():
        return await _run_batches_concurrently(
            batches=batches,
            model=model,
            concurrency=concurrency,
        )

    try:
        mapping, errors = asyncio.run(runner())
    except RuntimeError:
        import nest_asyncio
        nest_asyncio.apply()
        mapping, errors = asyncio.get_event_loop().run_until_complete(runner())

    print(f"Fallback batches: {errors} out of {len(batches)}")

    df[out_col] = df[text_col].map(mapping).fillna(df[text_col])
    return df

In [90]:
def create_clean_datafile(original_file_name, new_file_name):
    original_df = pd.read_csv(original_file_name, sep='\t',encoding='latin1',quoting=csv.QUOTE_NONE,names=['date','network','duration(s)','tag','text'],index_col=False)
    original_df["network"].value_counts()

    # original_df = original_df[:1000]

    new_english_list = []
    length = len(original_df)

    for r, row in original_df.iterrows():
        try:
            tokens = word_tokenize(row['text'])
            match = match_tokens(tokens, immigration_keywords)

            if match:
                new_row_data = {'date': row['date'], 'network': row['network'], 'duration(s)': row['duration(s)'], 'tag': row['tag'], 'text': str(row['text'])}
                new_english_list.append(new_row_data)

            
        except:
            continue


    new_df = pd.DataFrame(new_english_list)

    dropped_dups = new_df.drop_duplicates(subset=['text'], keep='first')

    # fox = dropped_dups[dropped_dups["network"] == "FOXNEWSW"].reset_index()
    # cnn = dropped_dups[dropped_dups["network"] == "CNNW"].reset_index()
    # msnbc = dropped_dups[dropped_dups["network"] == "MSNBCW"].reset_index()

    after_skrub = {"date": [], "network": [], "duration(s)": [], "tag": [], "text": [], "text_cleaned": []}

    sample_length = len(dropped_dups)
    chunk_size = 500
    current_index = 0

    while current_index <= sample_length - 1:
        subset = dropped_dups[current_index: current_index + chunk_size].reset_index()
        
        deduplicated_series = deduplicate(subset['text'])
        deduplicated_series.index = subset.index 
        subset['text_cleaned'] = deduplicated_series

        for i in range(len(subset)):
            after_skrub["date"].append(subset.loc[i]["date"])
            after_skrub["network"].append(subset.loc[i]["network"])
            after_skrub["duration(s)"].append(subset.loc[i]["duration(s)"])
            after_skrub["tag"].append(subset.loc[i]["tag"])
            after_skrub["text"].append(subset.loc[i]["text"])
            after_skrub["text_cleaned"].append(subset.loc[i]["text_cleaned"])

        current_index += chunk_size

    after_skrub = pd.DataFrame(after_skrub)

    dropped_again = after_skrub.drop_duplicates(subset=["text_cleaned"], keep="first").reset_index()
    
    return dropped_again


def run_llm(dropped_again, new_file_name):
    cleaned_df = clean_dataframe_with_llm_concurrent(
        dropped_again,
        text_col="text_cleaned",
        out_col="llm_text",
        batch_size=15,
        concurrency=20,
        model="gpt-4o-mini"
    )

    cleaned_df = cleaned_df.drop_duplicates(subset="llm_text", keep="first")
    cleaned_df.to_csv(new_file_name)

    return cleaned_df


In [83]:
import re
def remove_gibberish(cleaned_df):
    # Source - https://stackoverflow.com/a/39836090
    # Posted by MattR, modified by community. See post 'Timeline' for change history
    # Retrieved 2026-03-17, License - CC BY-SA 3.0

    new_df = {"date": [], "network": [], "duration(s)": [], "tag": [], "original_chyron": [], "text_cleaned": []}

    for i, row in cleaned_df.iterrows():
        chyron = row['text_cleaned']
        cleaned = re.sub(r'[^a-zA-Z0-9\s]', '', chyron)
        cleaned = re.sub(r'\s+', ' ', cleaned)
        new_df["date"].append(row["date"])
        new_df["network"].append(row["network"])
        new_df["duration(s)"].append(row["duration(s)"])
        new_df["tag"].append(row["tag"])
        new_df["original_chyron"].append(chyron)
        new_df["text_cleaned"].append(cleaned)

    return pd.DataFrame(new_df)

        

### 2018 - DONE

In [123]:
first_pass_2018 = create_clean_datafile("filtered_2018.tsv", "2018_clean.csv")

In [124]:
second_pass_2018 = remove_gibberish(first_pass_2018)

In [ ]:
run_llm(second_pass_2018, "2018_clean.csv")

### 2019 - DONE

In [126]:
first_pass_2019 = create_clean_datafile("filtered_2019.tsv", "2019_clean.csv")

In [127]:
second_pass_2019 = remove_gibberish(first_pass_2019)

In [128]:
run_llm(second_pass_2019, "2019_clean.csv")

100%|██████████| 1374/1374 [04:10<00:00,  5.48it/s]

Fallback batches: 0 out of 1374


,date,network,duration(s),tag,original_chyron,text_cleaned,llm_text
0,2019-01-01 01:11:40,BBCNEWS,3,BBCNEWS_20190101_010000_BBC_News/start/660,BEN BANG. Seeking Sanctuary migrant support group,BEN BANG Seeking Sanctuary migrant support group,BEN BANG Seeking Sanctuary migrant support group
1,2019-01-01 09:06:00,BBCNEWS,4,BBCNEWS_20190101_090000_BBC_News/start/360,NEW HORIZONS FLY-BY. Ukima Thule is a lump o( ...,NEW HORIZONS FLYBY Ukima Thule is a lump o ice...,
2,2019-01-01 11:09:01,FOXNEWSW,2,FOXNEWSW_20190101_110000_FOX__Friends/start/540,GOVE RNME NT. . TRUMP STANDS FIRM ON NEED FOR ...,GOVE RNME NT TRUMP STANDS FIRM ON NEED FOR BOR...,GOVERNMENT TRUMP STANDS FIRM ON NEED FOR BORDE...
3,2019-01-01 11:20:15,FOXNEWSW,9,FOXNEWSW_20190101_110000_FOX__Friends/start/1200,PARDONING ILLEGALS,PARDONING ILLEGALS,PARDONING ILLEGALS
4,2019-01-01 11:21:29,MSNBCW,6,MSNBCW_20190101_110000_Morning_Joe/start/1260,AP POLL: TOP 10 STORIES OF 2018. . #6: US. IMM...,AP POLL TOP 10 STORIES OF 2018 6 US IMMIGRATION,AP POLL TOP 10 STORIES OF 2018 6 US IMMIGRATION
...,...,...,...,...,...,...,...
20800,2019-12-31 11:23:38,MSNBCW,8,MSNBCW_20191231_110000_Morning_Joe/start/1380,"""MIGRATION T0 PRISON: AMERICA'S OBSESSION,WITH...",MIGRATION T0 PRISON AMERICAS OBSESSIONWITH LQC...,MIGRATION TO PRISON AMERICA'S OBSESSION WITH L...
20801,2019-12-31 11:25:41,MSNBCW,21,MSNBCW_20191231_110000_Morning_Joe/start/1500,THE ORIGINS OF U.S. IMMIGRATION PRISONS,THE ORIGINS OF US IMMIGRATION PRISONS,THE ORIGINS OF US IMMIGRATION PRISONS
20802,2019-12-31 11:26:55,MSNBCW,7,MSNBCW_20191231_110000_Morning_Joe/start/1560,COMPARING THE 0BAMA&TRUMPADMINISTRATIONSâ IM...,COMPARING THE 0BAMATRUMPADMINISTRATIONS IMMIGR...,COMPARING THE OBAMA TRUMP ADMINISTRATIONS IMMI...
20805,2019-12-31 19:44:33,FOXNEWSW,4,FOXNEWSW_20191231_190000_The_Daily_Briefing_Wi...,NEW IMMIGRATION LAWS TO TAKE EFFECT IN 2020,NEW IMMIGRATION LAWS TO TAKE EFFECT IN 2020,NEW IMMIGRATION LAWS TO TAKE EFFECT IN 2020


### 2020 - DONE

In [91]:
first_pass_2020 = create_clean_datafile("filtered_2020.tsv", "2020_clean.csv")

In [93]:
second_pass_2020 = remove_gibberish(first_pass_2020)

In [122]:
run_llm(second_pass_2020, "2020_clean.csv")

100%|██████████| 488/488 [01:03<00:00,  7.70it/s]

Fallback batches: 0 out of 488


,date,network,duration(s),tag,original_chyron,text_cleaned,llm_text
0,2020-01-01 01:13:49,BBCNEWS,2,BBCNEWS_20200101_010000_BBC_News/start/780,ICEIEDI'aUOI'lS,ICEIEDIaUOIlS,
6,2020-01-01 11:16:00,BBCNEWS,3,BBCNEWS_20200101_110000_BBC_News/start/960,Hong Kong protests. . Prokes'ers call for amne...,Hong Kong protests Prokesers call for amnesty ...,Protesters call for amnesty on thousands arres...
8,2020-01-01 18:43:25,CNNW,2,CNNW_20200101_184300_,SUPREME COURT FACES HOT-BU'ITON ISSUES IN 2020...,SUPREME COURT FACES HOTBUITON ISSUES IN 2020 C...,Cases involving immigration lie ahead
9,2020-01-01 19:48:00,FOXNEWSW,4,FOXNEWSW_20200101_194800_,MIGRANTS SEEKING U.S. ASYLUM WAIT IN MEXICO,MIGRANTS SEEKING US ASYLUM WAIT IN MEXICO,MIGRANTS SEEKING US ASYLUM WAIT IN MEXICO
18,2020-01-02 03:51:17,FOXNEWSW,2,FOXNEWSW_20200102_035100_,STUDENTS DEMAND UNIV REMOVE ICE JOB LISTINGS,STUDENTS DEMAND UNIV REMOVE ICE JOB LISTINGS,STUDENTS DEMAND UNIV REMOVE ICE JOB LISTINGS
...,...,...,...,...,...,...,...
7326,2020-12-29 10:20:04,BBCNEWS,2,BBCNEWS_20201229_100000_BBC_News/start/1200,Rohingya refugees moved. Some rights groups sa...,Rohingya refugees moved Some rights groups say...,Rohingya refugees moved. Some rights groups sa...
7327,2020-12-29 10:21:00,BBCNEWS,9,BBCNEWS_20201229_100000_BBC_News/start/1260,Rohingya refugees moved. Hundreds of thousands...,Rohingya refugees moved Hundreds of thousands ...,Rohingya refugees moved. Hundreds of thousands...
7339,2020-12-31 10:16:00,FOXNEWSW,8,FOXNEWSW_20201231_100000_FOX__Friends_First/st...,BIDEN T0 DELAY IMMIGRATION PLANS TO AVOID 'CRI...,BIDEN T0 DELAY IMMIGRATION PLANS TO AVOID CRISIS,BIDEN TO DELAY IMMIGRATION PLANS TO AVOID CRISIS
7341,2020-12-31 13:14:00,FOXNEWSW,7,FOXNEWSW_20201231_110000_FOX_and_Friends/start...,EL PASO BORDER CROSSING TEMPORARILY CLOSED,EL PASO BORDER CROSSING TEMPORARILY CLOSED,EL PASO BORDER CROSSING TEMPORARILY CLOSED


### 2021 - DONE

In [142]:
first_pass_2021 = create_clean_datafile("filtered_2021.tsv", "2021_clean.csv")

In [143]:
second_pass_2021 = remove_gibberish(first_pass_2021)

In [144]:
run_llm(second_pass_2021, "2021_clean.csv")

100%|██████████| 613/613 [02:15<00:00,  4.51it/s]

Fallback batches: 0 out of 613


,date,network,duration(s),tag,original_chyron,text_cleaned,llm_text
0,2021-01-01 04:27:01,FOXNEWSW,2,FOXNEWSW_20210101_040000_The_Greg_Gutfeld_Show...,ALIEN. TAPE,ALIEN TAPE,
5,2021-01-01 12:34:00,MSNBCW,8,MSNBCW_20210101_120000_MSNBC_Live/start/2040,TRUMP EXTENDS PANDEMIC-RELATEI] BANS 0N GREEN ...,TRUMP EXTENDS PANDEMICRELATEI BANS 0N GREEN CA...,TRUMP EXTENDS PANDEMIC RELATED BANS ON GREEN C...
10,2021-01-02 21:45:07,MSNBCW,2,MSNBCW_20210102_210000_MSNBC_Live_with_Yasmin_...,POLITICO: TRUMP ADMIN. MISSES DEADLINE FOR CEN...,POLITICO TRUMP ADMIN MISSES DEADLINE FOR CENSU...,POLITICO TRUMP ADMIN MISSES DEADLINE FOR CENSU...
16,2021-01-05 02:30:00,FOXNEWSW,2,FOXNEWSW_20210105_020000_Hannity/start/1800,PRESIDENT TRUMP MAKES THE CASE. FOR STRONG IMM...,PRESIDENT TRUMP MAKES THE CASE FOR STRONG IMMI...,PRESIDENT TRUMP MAKES THE CASE FOR STRONG IMMI...
23,2021-01-05 23:30:01,CNNW,2,CNNW_20210105_210000_Election_Day_in_America_G...,"FII'HI- nvvn Ulâ VV I mu UI'UI-I'I i'VVHI, J...",FIIHI nvvn Ul VV I mu UIUIII iVVHI JLIHI L IVI...,REMOVAL AND DETENTION NY DEMS BILL WOULD ALLOW...
...,...,...,...,...,...,...,...
9246,2021-12-31 01:50:00,FOXNEWSW,2,FOXNEWSW_20211231_010000_Tucker_Carlson_Tonigh...,"S "" %. USING UBERS AFTER CROSSING INTO THE U.S...",S USING UBERS AFTER CROSSING INTO THE US VAU O...,S USING UBERS AFTER CROSSING INTO THE US VAU O...
9247,2021-12-31 03:20:06,BBCNEWS,8,BBCNEWS_20211231_030000_BBC_News/start/1200,Migrants 1lown home. Bodies of 15 people kille...,Migrants 1lown home Bodies of 15 people killed...,Migrants flown home Bodies of 15 people killed...
9249,2021-12-31 06:43:14,FOXNEWSW,3,FOXNEWSW_20211231_060000_Tucker_Carlson_Tonigh...,BIDEN FLYING ILLEGAL IMMIGRANTS AROUND THE U.S.,BIDEN FLYING ILLEGAL IMMIGRANTS AROUND THE US,BIDEN FLYING ILLEGAL IMMIGRANTS AROUND THE US
9253,2021-12-31 12:09:00,FOXNEWSW,2,FOXNEWSW_20211231_110000_FOX_and_Friends/start...,N BN ! || P 0 R e. . BORDER AGENTS OVERWHELMED...,N BN P 0 R e BORDER AGENTS OVERWHELMED BY MIGR...,BORDER AGENTS OVERWHELMED BY MIGRANT SURGE


### 2022 - DONE

In [129]:
first_pass_2022 = create_clean_datafile("filtered_2022.tsv", "2022_clean.csv")

In [130]:
second_pass_2022 = remove_gibberish(first_pass_2022)

In [131]:
run_llm(second_pass_2022, "2022_clean.csv")

100%|██████████| 1192/1192 [04:22<00:00,  4.55it/s]

Fallback batches: 0 out of 1192


,date,network,duration(s),tag,original_chyron,text_cleaned,llm_text
0,2022-01-01 13:43:00,MSNBCW,2,MSNBCW_20220101_130000_Velshi/start/2580,For the acute treatment of migraine in adults ...,For the acute treatment of migraine in adults ...,
5,2022-01-01 21:04:55,BBCNEWS,2,BBCNEWS_20220101_210000_BBC_News/start/240,1taly migrants rescue. Ship carrying hundreds ...,1taly migrants rescue Ship carrying hundreds o...,Italy migrants rescue Ship carrying hundreds o...
6,2022-01-01 21:05:13,BBCNEWS,3,BBCNEWS_20220101_210000_BBC_News/start/300,1taly migrants rescue. 440 people on ship were...,1taly migrants rescue 440 people on ship were ...,Italy migrants rescue 440 people on ship were ...
7,2022-01-01 21:05:17,BBCNEWS,5,BBCNEWS_20220101_210000_BBC_News/start/300,1taly migrants rescue. Charity group Sea Watch...,1taly migrants rescue Charity group Sea Watch ...,Italy migrants rescue Charity group Sea Watch ...
8,2022-01-01 21:05:29,BBCNEWS,3,BBCNEWS_20220101_210000_BBC_News/start/300,Italy migrants rescue. Those onboard ship were...,Italy migrants rescue Those onboard ship were ...,Italy migrants rescue Those onboard ship were ...
...,...,...,...,...,...,...,...
18019,2022-12-31 16:23:00,FOXNEWSW,4,FOXNEWSW_20221231_160000_Cavuto_Live/start/1380,DHS WARNS OF POTENTIAL ATTACKS ON MIGRANTS -RPT,DHS WARNS OF POTENTIAL ATTACKS ON MIGRANTS RPT,DHS WARNS OF POTENTIAL ATTACKS ON MIGRANTS
18021,2022-12-31 18:51:00,MSNBCW,5,MSNBCW_20221231_180000_Alex_Witt_Reports/start...,DEPORTATION CASES IN U.S. JUMPED 29% IN FISCAL...,DEPORTATION CASES IN US JUMPED 29 IN FISCAL 2022,DEPORTATION CASES IN US JUMPED 29% IN FISCAL 2022
18022,2022-12-31 18:51:21,MSNBCW,22,MSNBCW_20221231_180000_Alex_Witt_Reports/start...,J TITLE 42 RULING LEAVES MIGRANTS IN LIMBO,J TITLE 42 RULING LEAVES MIGRANTS IN LIMBO,TITLE 42 RULING LEAVES MIGRANTS IN LIMBO
18023,2022-12-31 18:51:43,MSNBCW,19,MSNBCW_20221231_180000_Alex_Witt_Reports/start...,"MIGRANTS FEARING DEPORTATION, SLEEP ON EL PASO...",MIGRANTS FEARING DEPORTATION SLEEP ON EL PASO ...,MIGRANTS FEARING DEPORTATION SLEEP ON EL PASO ...


### 2023 - DONE

In [133]:
first_pass_2023 = create_clean_datafile("filtered_2023.tsv", "2023_clean.csv")

In [134]:
second_pass_2023 = remove_gibberish(first_pass_2023)

In [135]:
run_llm(second_pass_2023, "2023_clean.csv")

100%|██████████| 924/924 [03:14<00:00,  4.75it/s]

Fallback batches: 0 out of 924


,date,network,duration(s),tag,original_chyron,text_cleaned,llm_text
0,2023-01-01 03:51:00,CNNW,2,CNNW_20230101_010000_CNN_New_Years_Eve_Live_wi...,Y. W 2 /WEEEEEEEES. ployxy * ICEANI CRAADT. . T,Y W 2 WEEEEEEEES ployxy ICEANI CRAADT T,
3,2023-01-01 09:21:29,BBCNEWS,2,BBCNEWS_20230101_090000_BBC_News/start/1260,Australian Open. Novak Djokovic says he will '...,Australian Open Novak Djokovic says he will ne...,Australian Open Novak Djokovic says he will ne...
4,2023-01-01 09:21:37,BBCNEWS,3,BBCNEWS_20230101_090000_BBC_News/start/1260,Australian Open. Serb star was deported from t...,Australian Open Serb star was deported from th...,Australian Open Serb star was deported from th...
5,2023-01-01 11:29:43,FOXNEWSW,2,FOXNEWSW_20230101_110000_FOX_and_Friends_Sunda...,"BIDEN FACES BACKLASH OVER INFLATION, BORDER CR...",BIDEN FACES BACKLASH OVER INFLATION BORDER CRISIS,BIDEN FACES BACKLASH OVER INFLATION BORDER CRISIS
6,2023-01-01 11:38:29,FOXNEWSW,3,FOXNEWSW_20230101_110000_FOX_and_Friends_Sunda...,"""ROOT CAUSES OF MIGRATION""",ROOT CAUSES OF MIGRATION,ROOT CAUSES OF MIGRATION
...,...,...,...,...,...,...,...
13937,2023-12-31 21:06:39,FOXNEWSW,2,FOXNEWSW_20231231_210000_Fox_News_Live/start/360,HUGE MIGRANT CARAVAN HEADS TOWARDS THE U.S.,HUGE MIGRANT CARAVAN HEADS TOWARDS THE US,HUGE MIGRANT CARAVAN HEADS TOWARDS THE US
13938,2023-12-31 21:07:55,FOXNEWSW,2,FOXNEWSW_20231231_210000_Fox_News_Live/start/420,' CBP SOURCES: 286K+ MIGRANT ENCOUNTERS SINCE ...,CBP SOURCES 286K MIGRANT ENCOUNTERS SINCE DEC 1,CBP SOURCES 286K MIGRANT ENCOUNTERS SINCE DEC 1
13940,2023-12-31 22:27:39,CNNW,23,CNNW_20231231_220000_CNN_Newsroom_With_Jim_Aco...,TEXAS GOV. ABBOTT FLIES MORE THAN 350 MIGRANTS...,TEXAS GOV ABBOTT FLIES MORE THAN 350 MIGRANTS ...,TEXAS GOV ABBOTT FLIES MORE THAN 350 MIGRANTS ...
13941,2023-12-31 22:28:21,CNNW,21,CNNW_20231231_220000_CNN_Newsroom_With_Jim_Aco...,"TEXAS HAS BUSED MORE THAN 90,000 MIGRANTS TO C...",TEXAS HAS BUSED MORE THAN 90000 MIGRANTS TO CI...,TEXAS HAS BUSED MORE THAN 90000 MIGRANTS TO CI...


### 2024 - DONE

In [136]:
first_pass_2024 = create_clean_datafile("filtered_2024.tsv", "2024_clean.csv")

In [137]:
second_pass_2024 = remove_gibberish(first_pass_2024)

In [138]:
run_llm(second_pass_2024, "2024_clean.csv")

100%|██████████| 981/981 [03:54<00:00,  4.19it/s]

Fallback batches: 0 out of 981


,date,network,duration(s),tag,original_chyron,text_cleaned,llm_text
0,2024-01-01 00:15:19,FOXNEWSW,6,FOXNEWSW_20240101_000000_The_Big_Weekend_Show/...,"CBP: 300,000+ MIGRANT ENCOUNTERS IN DECEMBER |...",CBP 300000 MIGRANT ENCOUNTERS IN DECEMBER ioon,CBP 300000 MIGRANT ENCOUNTERS IN DECEMBER
1,2024-01-01 00:15:27,FOXNEWSW,8,FOXNEWSW_20240101_000000_The_Big_Weekend_Show/...,"CBP: 780,000 MIGRANT ENCOUNTERS IN FIRST QUARTER",CBP 780000 MIGRANT ENCOUNTERS IN FIRST QUARTER,CBP 780000 MIGRANT ENCOUNTERS IN FIRST QUARTER
2,2024-01-01 00:15:45,FOXNEWSW,3,FOXNEWSW_20240101_000000_The_Big_Weekend_Show/...,MASSIVE MIGRANT CARAVAN MOVES NORTH THROUGH ME...,MASSIVE MIGRANT CARAVAN MOVES NORTH THROUGH ME...,MASSIVE MIGRANT CARAVAN MOVES NORTH THROUGH ME...
3,2024-01-01 00:16:14,FOXNEWSW,2,FOXNEWSW_20240101_000000_The_Big_Weekend_Show/...,* BORDER AGENTS TAKEN OFF PATROLTO PROCESS MIG...,BORDER AGENTS TAKEN OFF PATROLTO PROCESS MIGRANTS,BORDER AGENTS TAKEN OFF PATROL TO PROCESS MIGR...
4,2024-01-01 09:13:23,BBCNEWS,5,BBCNEWS_20240101_090000_BBC_News/start/780,EU visa restrictions lifted for Kosovo,EU visa restrictions lifted for Kosovo,
...,...,...,...,...,...,...,...
14879,2024-12-31 15:22:48,CNNW,2,CNNW_20241231_150000_New_Years_Eve_Live_Around...,visa @D E mmz-l,visa D E mmzl,visa
14880,2024-12-31 15:38:26,FOXNEWSW,8,FOXNEWSW_20241231_150000_Americas_Newsroom/sta...,DEPORTATION DELAY,DEPORTATION DELAY,DEPORTATION DELAY
14882,2024-12-31 15:48:00,FOXNEWSW,2,FOXNEWSW_20241231_150000_Americas_Newsroom/sta...,FOX. . ICE ENDS 2 PROGRAMS OFFERING ILLEGAL MI...,FOX ICE ENDS 2 PROGRAMS OFFERING ILLEGAL MIGRA...,ICE ENDS 2 PROGRAMS OFFERING ILLEGAL MIGRANTS ...
14883,2024-12-31 17:17:17,FOXNEWSW,2,FOXNEWSW_20241231_170000_Outnumbered/start/1020,RICK WILSON INSULTS TRUMP VOTERS OVER IMMIGRATION,RICK WILSON INSULTS TRUMP VOTERS OVER IMMIGRATION,INSULTS TRUMP VOTERS OVER IMMIGRATION


### 2025 - DONE

In [139]:
first_pass_2025 = create_clean_datafile("filtered_2025.tsv", "2025_clean.csv")

In [140]:
second_pass_2025 = remove_gibberish(first_pass_2025)

In [141]:
run_llm(second_pass_2025, "2025_clean.csv")

100%|██████████| 1818/1818 [07:03<00:00,  4.30it/s]

Fallback batches: 0 out of 1818


,date,network,duration(s),tag,original_chyron,text_cleaned,llm_text
0,2025-01-01 00:33:38,MSNBCW,24,MSNBCW_20241231_230000_Alex_Wagner_Tonight/sta...,MIGRANT VICTIMS OF SEXUAL ASSAULT. FACE HURDLE...,MIGRANT VICTIMS OF SEXUAL ASSAULT FACE HURDLES...,MIGRANT VICTIMS OF SEXUAL ASSAULT FACE HURDLES...
1,2025-01-01 01:01:32,MSNBCW,11,MSNBCW_20250101_010000_All_In_With_Chris_Hayes...,MUSK & RAMASWAMY EXPOSE GOP RIFT. OVER IMMIGRA...,MUSK RAMASWAMY EXPOSE GOP RIFT OVER IMMIGRANT ...,MUSK RAMASWAMY EXPOSE GOP RIFT OVER IMMIGRANT ...
2,2025-01-01 01:09:06,MSNBCW,18,MSNBCW_20250101_010000_All_In_With_Chris_Hayes...,. MUSK CLASHES WITH MAGA BASE OVER SKILLED IMM...,MUSK CLASHES WITH MAGA BASE OVER SKILLED IMMIG...,MUSK CLASHES WITH MAGA BASE OVER SKILLED IMMIG...
3,2025-01-01 01:30:00,MSNBCW,2,MSNBCW_20250101_010000_All_In_With_Chris_Hayes...,promo purpa. in-app purcha. . ins. Contain:,promo purpa inapp purcha ins Contain,
4,2025-01-01 09:10:58,BBCNEWS,3,BBCNEWS_20250101_090000_BBC_News/start/600,Big rise in 2024 migrant Channel crossings,Big rise in 2024 migrant Channel crossings,Big rise in 2024 migrant Channel crossings
...,...,...,...,...,...,...,...
27607,2025-12-28 22:26:45,FOXNEWSW,2,FOXNEWSW_20251228_220000_The_Big_Weekend_Show/...,> TODD LYONS | ACTING ICE DIRECTOR T. ICE UNDE...,TODD LYONS ACTING ICE DIRECTOR T ICE UNDER ATTACK,TODD LYONS ACTING ICE DIRECTOR T ICE UNDER ATTACK
27608,2025-12-28 22:28:22,FOXNEWSW,2,FOXNEWSW_20251228_220000_The_Big_Weekend_Show/...,STOKING HATRED. CHICAGO MAYOR CLAIMS ICE AGENT...,STOKING HATRED CHICAGO MAYOR CLAIMS ICE AGENTS...,STOKING HATRED CHICAGO MAYOR CLAIMS ICE AGENTS...
27609,2025-12-28 22:29:10,FOXNEWSW,3,FOXNEWSW_20251228_220000_The_Big_Weekend_Show/...,ANTI-ICE MESSAGE. WALZ'S DAUGHTER SAYS ICE 'TE...,ANTIICE MESSAGE WALZS DAUGHTER SAYS ICE TERROR...,ANTIICE MESSAGE WALZS DAUGHTER SAYS ICE TERROR...
27611,2025-12-29 20:13:54,BBCNEWS,8,BBCNEWS_20251229_200000_BBC_News/start/780,Calls grow for Fattah's deportation,Calls grow for Fattahs deportation,Calls grow for Fattahs deportation


### 2026 - need file

In [145]:
first_pass_2026 = create_clean_datafile("filtered_2026.tsv", "2026_clean.csv")

In [146]:
second_pass_2026 = remove_gibberish(first_pass_2026)

In [147]:
run_llm(second_pass_2026, "2026_clean.csv")

100%|██████████| 381/381 [01:16<00:00,  4.98it/s]

Fallback batches: 0 out of 381


,date,network,duration(s),tag,original_chyron,text_cleaned,llm_text
0,2026-01-07 20:48:00,BBCNEWS,13,BBCNEWS_20260107_203000_The_Context/start/1080,US ICE immigration agent shoots woman dead,US ICE immigration agent shoots woman dead,US ICE immigration agent shoots woman dead
1,2026-01-07 20:49:00,CNNW,61,CNNW_20260107_200000_CNN_News_Central/start/2940,"MN GOV WALZ: ""I'M ANGRY"" ABOUT ICE KILLING OF ...",MN GOV WALZ IM ANGRY ABOUT ICE KILLING OF WOMAN,MN GOV WALZ IM ANGRY ABOUT ICE KILLING OF WOMAN
2,2026-01-07 20:49:00,FOXNEWSW,2,FOXNEWSW_20260107_200000_The_Story_With_Martha...,> TIM WALZ (D) | MINNESOTA GOVERNOR -. . NN HI...,TIM WALZ D MINNESOTA GOVERNOR NN HIDRATE AN MN...,
3,2026-01-07 20:50:58,CNNW,4,CNNW_20260107_200000_CNN_News_Central/start/3000,MN GOV WALZ URGES PROTESTERS O REMAIN PEACEFUL...,MN GOV WALZ URGES PROTESTERS O REMAIN PEACEFUL...,MN GOV WALZ URGES PROTESTERS TO REMAIN PEACEFU...
4,2026-01-07 20:51:42,MSNOW,2,MSNOW_20260107_190000_Katy_Tur_Reports/start/6660,NOW: MN GOV. TIM WALZ SPEAKS ON FATAL ICE SHOO...,NOW MN GOV TIM WALZ SPEAKS ON FATAL ICE SHOOTI...,NOW MN GOV TIM WALZ SPEAKS ON FATAL ICE SHOOTING
...,...,...,...,...,...,...,...
5754,2026-02-27 23:53:54,MSNOW,2,MSNOW_20260227_230000_The_Beat_With_Ari_Melber...,GOURTS GHECK SOME OF TRUMP'S ICE TACTICS EE&A&...,GOURTS GHECK SOME OF TRUMPS ICE TACTICS EEA 1I...,GOURTS GHECK SOME OF TRUMPS ICE TACTICS EEA 1I...
5755,2026-02-28 00:25:50,BBCNEWS,4,BBCNEWS_20260228_000000_BBC_News/start/1500,Iran after latest talks on immigration reforms...,Iran after latest talks on immigration reforms...,Iran after latest talks on immigration reforms...
5756,2026-02-28 04:15:16,FOXNEWSW,3,FOXNEWSW_20260228_040000_Fox_News_at_Night/sta...,C; BILL WOULD RESTORE BENEFITS FOR ILLEGAL ALIENS,C BILL WOULD RESTORE BENEFITS FOR ILLEGAL ALIENS,BILL WOULD RESTORE BENEFITS FOR ILLEGAL ALIENS
5757,2026-02-28 04:19:48,FOXNEWSW,4,FOXNEWSW_20260228_040000_Fox_News_at_Night/sta...,U.S. OLYMPIAN DEFENDS LEGAL IMMIGRATION,US OLYMPIAN DEFENDS LEGAL IMMIGRATION,US OLYMPIAN DEFENDS LEGAL IMMIGRATION
